# 02 - Distribution Shift Characterization

**Project:** Trustworthy AMI Forecasting Under Behavioral Distribution Shift (`shift-ami`)  
**Objective:** Quantify model-independent statistical distribution shifts (moments, 1D Wasserstein distance, Kolmogorov-Smirnov two-sample tests) between calibration and shifted test regimes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shift_ami.config import load_config
from shift_ami.evaluation.shift_analysis import compute_distribution_shift_moments

config = load_config("../configs/smoke.yaml")
df = pd.read_parquet(config.paths.processed_dir / "cohort_total_halfhourly.parquet")

# Split into Reference (Pre-Shift / Calibration) vs Test (Shifted)
ref_mask = (df['timestamp'] >= '2013-01-01') & (df['timestamp'] <= '2013-03-31')
test_mask = (df['timestamp'] >= '2013-05-16') & (df['timestamp'] <= '2013-06-30')

ref_series = df.loc[ref_mask, 'energy_kwh'].values
test_series = df.loc[test_mask, 'energy_kwh'].values

shift_diag = compute_distribution_shift_moments(ref_series, test_series)
for k, v in shift_diag.items():
    print(f"{k:25s}: {v}")

## 2. Pre vs Post Shift Density Comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(ref_series, bins=50, density=True, alpha=0.6, label='Reference Period (Winter/Spring)', color='#1f77b4')
plt.hist(test_series, bins=50, density=True, alpha=0.6, label='Evaluation Period (Summer / Shift)', color='#ff7f0e')
plt.xlabel('Load (kWh / 30-min)')
plt.ylabel('Empirical Density')
plt.title(f'Distribution Shift: Wasserstein-1 Distance = {shift_diag["wasserstein_1d"]:.4f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()